# Optimisation

5 plis pour valider et 3 plis internes pour regler les parametres. Choisir le modele avant le test. Projet pedagogique, sans usage clinique.

## 1. Donnees

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.optimize import optimize_model, cross_validate_model
from src.evaluate import evaluate_model

folder = PROJECT_ROOT / "metrique"
folder.mkdir(exist_ok=True)
models_folder = PROJECT_ROOT / "models"
models_folder.mkdir(exist_ok=True)

df = pd.read_csv(PROJECT_ROOT / "data/raw/student_mental_health.csv")
df = df.drop_duplicates().dropna(subset=["Mental_Health_Risk"])
X = df.drop(columns=["Student_ID", "Mental_Health_Risk"])
y = df["Mental_Health_Risk"]
X_train, X_test, y_train_text, y_test_text = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train_text)
class_names = encoder.classes_
assert set(X_train.index).isdisjoint(X_test.index)
print("Train :", len(X_train), "- Test reserve :", len(X_test))


Train : 1600 - Test reserve : 400


## 2. Parametres a essayer

GridSearchCV choisit le meilleur F1 macro.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, 
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        random_state=42, 
        n_jobs=2
    ),
    "XGBoost": XGBClassifier(
        learning_rate=0.05, 
        subsample=0.8, 
        colsample_bytree=0.8,
        objective="multi:softprob", 
        eval_metric="mlogloss",
        tree_method="hist", 
        random_state=42, 
        n_jobs=2,
    ),
}
parameters = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10]
        },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [8, None],
        "model__min_samples_leaf": [2],
    },
    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [2, 3],
    },
}
validation = {}
